# Attention
I found the following answer from https://gemini.google.com/ is helpful to understand what is basic attention

Think of it like this: for each word in the input sequence, the attention mechanism wants to figure out how relevant every other word (including itself) is to it. To do this, it uses three different perspectives on each word, represented by these three vectors.

Here's a breakdown of each:

### 1. Query (Q): What am I looking for?

- The query vector represents the word (or token) for which we are currently trying to find relevant information from the other words in the sequence.
- You can think of the query as your "search query." If you're trying to understand the word "it" in a sentence, the query vector for "it" will represent what kind of information we need to find from the rest of the sentence to understand "it."

#### 2. Key (K): What information do I contain?

- The key vector represents another word (or token) in the sequence and acts like a label or identifier for the information that this word contains.
- You can think of the key as the "keywords" or "tags" associated with each piece of information in your sequence. When our query looks at other words, it compares its "search query" (Q) with the "keywords" (K) of those other words to see if they are relevant.

### 3. Value (V): What is the actual information I offer?

- The value vector represents the actual information associated with a word (or token) that we might want to "attend to" if its key is found to be relevant to our query.
- You can think of the value as the actual "content" or "data" that a word holds. If a word's key matches our query well, then we "retrieve" its value.

### Analogy: Searching a Database

Imagine you have a database of documents, and you want to find information relevant to a specific topic:

- Your Query (Q) is your search term (e.g., "artificial intelligence").
- Each document in the database has a set of Keys (K) or tags associated with it (e.g., "machine learning," "neural networks," "robotics").
- The Value (V) is the actual content of each document.
- When you perform a search, your query is compared to the keys of all the documents. Documents whose keys are highly relevant to your query are considered important, and you might then look at their content (the value).

In the Transformer's attention mechanism:

- For each word in the input sequence, we generate a query vector.
- For every word in the input sequence (including the word itself), we generate a key vector and a value vector.
- The query vector of a word is compared to the key vectors of all other words to determine how much attention should be paid to each of those words.
- The value vectors of the words that receive high attention are then combined (weighted by the attention scores) to produce the output of the attention mechanism for the original word.

### How are Q, K, and V obtained?

For each input token (after embedding and positional encoding), three different weight matrices (W_Q, W_K, W_V) are learned during the training process. The embedding vector of the token is then multiplied by each of these matrices to produce the query, key, and value vectors, respectively:

- Query (Q) = Input Embedding * W_Q
- Key (K) = Input Embedding * W_K
- Value (V) = Input Embedding * W_V

These weight matrices allow the model to learn different aspects of each word that are relevant for different kinds of comparisons when calculating attention.

So, in essence, the attention mechanism uses these query, key, and value vectors to figure out for each word: "What am I looking for?" (Query), "What information does each other word contain?" (Key), and "What is the actual information I should pay attention to?" (Value).

![attention](./resources/attention.png)

In [1]:
import torch

d_model = 5  # Dimension of the embeddings
d_k = 3      # Dimension of the W_K and W_Q vectors
d_v = 4      # Dimension of the W_V vectors

# Assume we have three embeddings, each has dimension d_model
E1 = torch.rand(d_model)
E2 = torch.rand(d_model)
E3 = torch.rand(d_model)
E = torch.stack([E1, E2, E3])

# Assume we have the query weights matrix with size (d_model, d_k)
W_Q = torch.rand([d_model, d_k])
# Assume we have the key weights matrix with size (d_model, d_k)
W_K = torch.rand([d_model, d_k])
# Assume we have the value weights matrix with size (d_model, d_v)
W_V = torch.rand([d_model, d_v])

print(W_Q)
print(W_K)
print(W_V)

tensor([[0.3446, 0.5839, 0.9604],
        [0.7692, 0.9039, 0.0605],
        [0.4528, 0.7341, 0.2898],
        [0.9941, 0.9434, 0.3518],
        [0.9862, 0.6058, 0.4737]])
tensor([[0.5240, 0.9020, 0.4636],
        [0.5041, 0.0966, 0.2322],
        [0.3486, 0.2957, 0.6466],
        [0.2529, 0.8873, 0.0675],
        [0.5985, 0.0117, 0.6982]])
tensor([[0.6507, 0.4130, 0.6883, 0.2279],
        [0.5148, 0.4390, 0.1860, 0.7307],
        [0.9504, 0.0423, 0.2101, 0.1786],
        [0.0868, 0.9614, 0.9868, 0.3077],
        [0.4609, 0.2190, 0.0794, 0.8517]])


In the above initialization, we assume that we have 3 embeddings with dimension `d_model`. Then, we initialize random matrices for `W_Q`, `W_K`, and `W_V`:
- `W_Q` has dimension `(d_model, d_k)` because we will do `Ei @ W_Q` that outputs a vector `Q` with size `d_k`
- `W_K` has dimension `(d_model, d_k)` because we will do `Ei @ W_K` that outputs a vector `K` with size `d_k`
- `W_V` has dimension `(d_model, d_v)` because we will do `Ei @ W_V` that outputs a vector `V` with size `d_v`

Let's compute Q, K, and V for each embedding.

In [2]:
Q = E @ W_Q
K = E @ W_K
V = E @ W_V

print(Q)
print(K)
print(V)

tensor([[1.1320, 1.1905, 0.7435],
        [1.8997, 1.8092, 0.6878],
        [2.3760, 2.2377, 1.0493]])
tensor([[0.6826, 0.8306, 0.6152],
        [1.2124, 0.3068, 1.2292],
        [1.1780, 1.1828, 1.2195]])
tensor([[0.7642, 0.7514, 0.8108, 0.6761],
        [1.4173, 0.6574, 0.4034, 1.5540],
        [1.3239, 1.3092, 1.2873, 1.3695]])


For the above operations, you can understand it as: given a query, and for each key, how much attention should we pay on its corresponding value?

The formula of the attention score is: $\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$. After computing the attention, we will then multiply with the embeddings to create new representation of the tokens, which involve context from other tokens.

For simplicity, we will compute the attention by using matrix operation rather than doing it step by step:


In [3]:
import math

# Dimension:
# A = softmax(Q (d_model, d_k) @ K.T (d_k, d_model``) / sqrt(d_k)) -> (d_model, d_model)
# B = A @ V (d_model, d_v) -> (d_model, d_v)
attention = torch.softmax(Q @ K.T / math.sqrt(d_k), dim=1) @ V
print(attention)

tensor([[1.2276, 1.0038, 0.9345, 1.2695],
        [1.2443, 1.0505, 0.9875, 1.2874],
        [1.2644, 1.0878, 1.0270, 1.3107]])


# Multi-head Attention
Multi-head attention is an extension of the attention mechanism that allows the model to focus on different parts of the input sequence simultaneously. Instead of computing a single attention score for each query-key pair, multi-head attention computes multiple sets of attention scores (or "heads") in parallel, each with its own learned weight matrices. The outputs of these heads are then concatenated and linearly transformed to produce the final output.

You can just understand multi-head attention as linearly multiple single-head attention. As a standard practice, let's say we have `h` head attention, we choose the following variables:
- `d_k` is usually equal `d_v`
- `d_k = d_v = d_model / h`

That means, given that the embedding dimension is `d_model = 8`, if we want to have `h = 2` heads attention, then we choose `d_k = d_v = 8 / 2 = 4`
First head:
- `W_K_1` has dimension `(n, d_k) = (3, 4)`
- `W_Q_1` has dimension `(n, d_k) = (3, 4)`
- `W_V_1` has dimension `(n, d_v) = (3, 4)`

Second head:
- `W_K_2` has dimension `(n, d_k) = (3, 4)`
- `W_Q_2` has dimension `(n, d_k) = (3, 4)`
- `W_V_2` has dimension `(n, d_v) = (3, 4)`


In [4]:
d_model = 8             # Dimension of the embeddings
h = 2                   # 2-head attention
d_k = int(d_model / h)  # Dimension of the W_K and W_Q vectors, d_model / h
d_v = d_k               # Dimension of the W_V vectors, same as d_k

# Assume we have three embeddings, each has dimension d_model
E1 = torch.rand(d_model)
E2 = torch.rand(d_model)
E3 = torch.rand(d_model)
E = torch.stack([E1, E2, E3])

# First head
W_Q_1 = torch.rand([d_model, d_k])
W_K_1 = torch.rand([d_model, d_k])
W_V_1 = torch.rand([d_model, d_v])

# Second head
W_Q_2 = torch.rand([d_model, d_k])
W_K_2 = torch.rand([d_model, d_k])
W_V_2 = torch.rand([d_model, d_v])

# Calculate Q, K, V for the first head
Q_1 = E @ W_Q_1
K_1 = E @ W_K_1
V_1 = E @ W_V_1

# Calculate Q, K, V for the second head
Q_2 = E @ W_Q_2
K_2 = E @ W_K_2
V_2 = E @ W_V_2

# Compute the final output for each head
O_1 = torch.softmax(Q_1 @ K_1.T / math.sqrt(d_k), dim=1) @ V_1
O_2 = torch.softmax(Q_2 @ K_2.T / math.sqrt(d_k), dim=1) @ V_2
print(O_1)
print(O_2)

tensor([[3.6605, 2.5474, 3.1241, 3.9401],
        [3.5948, 2.5029, 3.0556, 3.8401],
        [3.7044, 2.5782, 3.1725, 4.0105]])
tensor([[2.1100, 2.9937, 3.7841, 3.3814],
        [2.0812, 2.9690, 3.7436, 3.3475],
        [2.1119, 2.9954, 3.7870, 3.3837]])


Now that we have two output matrices `O1` and `O2` from 2 heads, we will need to combine them into one. Usually done by:
1. Concat `O1` and `O2` by `O = [O1 || O2]`, that is, if `O1 = [[1, 2], [3, 4]]`, `O2 = [[5, 6], [7, 8]]`, then `O = [O1 || O2] = [[1, 2, 5, 6], [3, 4, 7, 8]]`
2. Multiply the output matrix `O` with a learnable matrix `W_O`, where `W_O` has size `(h * d_v, d_model)`. In this case, size of `W_O` is `(8, 8)`

In [5]:
# Initialize a random matrix W_O with size (h * d_v, d_model)
W_O = torch.rand([h * d_v, d_model])

# Concat O1 and O2
O = torch.cat((O_1, O_2), dim=1)
print(O)

# Multiply O with W_O
O @ W_O

tensor([[3.6605, 2.5474, 3.1241, 3.9401, 2.1100, 2.9937, 3.7841, 3.3814],
        [3.5948, 2.5029, 3.0556, 3.8401, 2.0812, 2.9690, 3.7436, 3.3475],
        [3.7044, 2.5782, 3.1725, 4.0105, 2.1119, 2.9954, 3.7870, 3.3837]])


tensor([[16.3182, 14.0281, 11.5710, 13.5217, 15.2297, 12.5915,  9.4499, 16.2082],
        [16.0656, 13.8179, 11.3957, 13.2838, 14.9884, 12.3926,  9.2819, 15.9275],
        [16.4450, 14.1327, 11.6522, 13.6582, 15.3535, 12.6832,  9.5445, 16.3593]])

# Masked Multi-head Attention
Masked Multi-Head Attention is a variation of Multi-Head Attention where the attention mechanism is prevented from attending to subsequent tokens in the sequence. In tasks where the model needs to predict the next token based on the previous tokens (this is called autoregressive generation, which is how models like GPT work and how the decoder in a translation Transformer works), the model should only have access to the information available up to the current token's position.

In [10]:
example = torch.rand([4, 4])
print(example)

mask = torch.tensor([
    [0, -float("inf"), -float("inf"), -float("inf")],
    [0, 0, -float("inf"), -float("inf")],
    [0, 0, 0, -float("inf")],
    [0, 0, 0, 0]
])
print(mask)

# This step should be done before softmax
masked_example = example + mask
print(masked_example)

# Apply softmax
torch.softmax(masked_example, dim=1)

tensor([[0.4193, 0.5688, 0.5359, 0.4636],
        [0.8167, 0.0630, 0.9473, 0.0019],
        [0.9855, 0.1704, 0.0974, 0.2490],
        [0.1761, 0.3784, 0.5541, 0.5404]])
tensor([[0., -inf, -inf, -inf],
        [0., 0., -inf, -inf],
        [0., 0., 0., -inf],
        [0., 0., 0., 0.]])
tensor([[0.4193,   -inf,   -inf,   -inf],
        [0.8167, 0.0630,   -inf,   -inf],
        [0.9855, 0.1704, 0.0974,   -inf],
        [0.1761, 0.3784, 0.5541, 0.5404]])


tensor([[1.0000, 0.0000, 0.0000, 0.0000],
        [0.6800, 0.3200, 0.0000, 0.0000],
        [0.5394, 0.2387, 0.2219, 0.0000],
        [0.1952, 0.2390, 0.2849, 0.2810]])

Because the softmax function involves `e`, where `e^0` will become 1, and `e^-inf` will become very close to 0. Therefore, we use 0 to represents the parts that are visible to the model, `-inf` to represent the parts that are not visible.